### Bype-Pair Encoding (BPE) token learner

__Algorithm__

`Repeat`  
- choose most frequent neighboring pair ('a', 'b')
- add a new merged symbol ('ab') to the vocabulary
- replace every 'a' 'b' in the corpus with 'ab'    

`Until k merges`

__superword__

In [ ]:
import re

raw_text = """
A house is a single-unit's residential building. It may range in complexity from a rudimentary hut to a complex structure of wood, masonry, concrete or other material, outfitted with plumbing, electrical, and heating, ventilation, and air conditioning systems.[1][2] Houses use a range of different roofing systems to keep precipitation such as rain from getting into the dwelling space. Houses generally have doors or locks to secure the dwelling space and protect its inhabitants and contents from burglars or other trespassers. Most conventional modern houses in Western cultures will contain one or more bedrooms and bathrooms, a kitchen or cooking area, and a living room. A house may have a separate dining room, or the eating area may be integrated into the kitchen or another room. Some large houses in North America have a recreation room. In traditional agriculture-oriented societies, domestic animals such as chickens or larger livestock (like cattle) may share part of the house with humans.
The social unit that lives in a house is known as a household. Most commonly, a household is a family unit of some kind, although households may also have other social groups, such as roommates or, in a rooming house, unconnected individuals, that typically use a house as their home. Some houses only have a dwelling space for one family or similar-sized group; larger houses called townhouses or row houses may contain numerous family dwellings in the same structure. A house may be accompanied by outbuildings, such as a garage for vehicles or a shed for gardening equipment and tools. A house may have a backyard, a front yard or both, which serve as additional areas where inhabitants can relax, eat, or exercise.
"""

text = raw_text.lower()

# pretokenization
pattern = re.compile(
	# Contractions: common English apostrophe suffixes
	r"'s|'t|'re|'ve|'m|'ll|'d|"
	# Sequence of English lowercase letters (after optional space)
	r" ?[a-z]+|"
	# Sequence of digits (after optional space)
	r" ?\d+|"
	# Punctuation: anything that isn't a space, letter, or digit (after optional space)
	r" ?[^\s\d[a-z]]+|"
	# Whitespace
	r"\s+(?!\S)|\s+" 
)

tokens = pattern.findall(text)
print(tokens)

['\n', 'a', ' house', ' is', ' a', ' single', 'unit', "'s", ' residential', ' building', ' it', ' may', ' range', ' in', ' complexity', ' from', ' a', ' rudimentary', ' hut', ' to', ' a', ' complex', ' structure', ' of', ' wood', ' masonry', ' concrete', ' or', ' other', ' material', ' outfitted', ' with', ' plumbing', ' electrical', ' and', ' heating', ' ventilation', ' and', ' air', ' conditioning', ' systems', '1', '2', ' houses', ' use', ' a', ' range', ' of', ' different', ' roofing', ' systems', ' to', ' keep', ' precipitation', ' such', ' as', ' rain', ' from', ' getting', ' into', ' the', ' dwelling', ' space', ' houses', ' generally', ' have', ' doors', ' or', ' locks', ' to', ' secure', ' the', ' dwelling', ' space', ' and', ' protect', ' its', ' inhabitants', ' and', ' contents', ' from', ' burglars', ' or', ' other', ' trespassers', ' most', ' conventional', ' modern', ' houses', ' in', ' western', ' cultures', ' will', ' contain', ' one', ' or', ' more', ' bedrooms', ' and

In [36]:
# BPE functions

def get_init_vocab(tokens):
	vocab ={}
	for token in tokens:
		space_word = " ".join(list(token)) + " </w>"
		vocab[space_word] = vocab.get(space_word, 0) + 1
	return vocab

def get_stats(vocab):
	pairs = {}
	for word, freq in vocab.items():
		sym = word.split()
		word_pairs = [(sym[i], sym[i+1]) for i in range(len(sym)-1)]
		for pair in word_pairs:
			pairs[pair] = pairs.get(pair, 0) + freq
	return pairs

def merge_vocab(best_pair, vocab_in):
	vocab_out = {}
	merged_pair = best_pair.replace(' ', '') 

	escaped_pair = re.escape(best_pair)

	pattern = re.compile(r'(?<!\S)' + escaped_pair + r'(?!\S)') # only match 'a b' if it is alone, not 'c a b'

	for word, freq in vocab_in.items():
		if best_pair in word:
			new_word = pattern.sub(merged_pair, word)
			vocab_out[new_word] = freq
		else:
			vocab_out[word] = freq
	return vocab_out

def train_bpe(raw_text, num_merges):
	# Execute the text pipeline in order
	text = raw_text.lower()
	tokens = re.findall(r'\w+|[^\w\s]', text) 
	
	# Initialize vocabulary with pretokenized chunks
	vocab = get_init_vocab(tokens)
	
	# Run the BPE training loop
	for i in range(num_merges):
		pairs = get_stats(vocab)
		if not pairs:
			break
			
		best_pair_tuple = max(pairs, key=pairs.get)
		best_pair = ' '.join(best_pair_tuple)
		
		vocab = merge_vocab(best_pair, vocab) 
		
		merged_string = best_pair.replace(' ', '')
		print(f"Merge {i+1:02d}: '{best_pair}' -> '{merged_string}'")
		
	return vocab

# generate token ids
def create_token_ids(final_vocabs):
	# 1. gather all unique subwords
	unique_subwords = set()
	for word in final_vocabs.keys():
		for subword in word.split():
			unique_subwords.add(subword)
			
	sorted_subwords = sorted(list(unique_subwords)) # list

	# string to number encoder - enumerate(list)
	token_to_id = {subword : token_id for token_id, subword in enumerate(sorted_subwords)}
	# number to string decoder - dictionary.items()
	id_to_token = {token_id : subword for subword, token_id in token_to_id.items()} 

	return token_to_id, id_to_token


# Run the pipeline
final_vocab = train_bpe(raw_text, num_merges=10)

token_to_id, id_to_token = create_token_ids(final_vocab)



Merge 01: 'e </w>' -> 'e</w>'
Merge 02: 's </w>' -> 's</w>'
Merge 03: 'i n' -> 'in'
Merge 04: 'a </w>' -> 'a</w>'
Merge 05: 'r </w>' -> 'r</w>'
Merge 06: 'o u' -> 'ou'
Merge 07: ', </w>' -> ',</w>'
Merge 08: 'd </w>' -> 'd</w>'
Merge 09: 'h ou' -> 'hou'
Merge 10: 'o m' -> 'om'


In [37]:
# sort by frequency (x[1]) in descending order
for word, freq in sorted(final_vocab.items(), key=lambda x:x[1], reverse=True):
	print(f"{word} : {freq}")

print(final_vocab)

a</w> : 23
,</w> : 22
o r</w> : 14
. </w> : 13
hou s e</w> : 8
m a y </w> : 8
in </w> : 7
a n d</w> : 7
hou s e s</w> : 7
a s</w> : 7
t h e</w> : 7
h a v e</w> : 6
o f </w> : 4
s u c h </w> : 4
r o om </w> : 4
i s</w> : 3
- </w> : 3
u n i t </w> : 3
f r om </w> : 3
t o </w> : 3
o t h e r</w> : 3
d w e l l in g </w> : 3
s p a c e</w> : 3
s om e</w> : 3
f a m i l y </w> : 3
f o r</w> : 3
r a n g e</w> : 2
s t r u c t u r e</w> : 2
w i t h </w> : 2
s y s t e m s</w> : 2
[ </w> : 2
] </w> : 2
u s e</w> : 2
in t o </w> : 2
in h a b i t a n t s</w> : 2
m o s t </w> : 2
c o n t a in </w> : 2
o n e</w> : 2
k i t c h e n </w> : 2
a r e a</w> : 2
b e</w> : 2
l a r g e r</w> : 2
s o c i a l </w> : 2
t h a t </w> : 2
hou s e h o l d</w> : 2
s in g l e</w> : 1
' </w> : 1
s</w> : 1
r e s i d e n t i a l </w> : 1
b u i l d in g </w> : 1
i t </w> : 1
c om p l e x i t y </w> : 1
r u d i m e n t a r y </w> : 1
h u t </w> : 1
c om p l e x </w> : 1
w o o d</w> : 1
m a s o n r y </w> : 1
c o n c r e t e</w